In [ ]:
#Helper

def get_response_text(response_dict):

    if not isinstance(response_dict, dict):
        return ""
    return (response_dict.get("text") or "").strip()

In [ ]:
#ask for an answer, then ask the model "is this answer correct? if not, fix it." Two API calls total.

def solve_with_reflection(question, model=None, temperature=0.0, too_wordy=False):



    first_system = (
        "You are great at finding the correct answers to any given questions. "
        "Solve the user's question and reply with the best final answer you can."
    )

    first_prompt = f"""Question: {question} What is the answer?"""
    first_response = call_model_chat_completions(
        first_prompt,
        system=first_system,
        model=model,
        temperature=temperature,
    )
    initial_answer = get_response_text(first_response)



    reflection_system = (
        "You are great at reviewing questions. "
        "Check whether the proposed answer is correct and complete. "
        "If it is wrong, fix it. "
        "Return only the correct and complete final answer."
    )

    reflection_prompt = f"""You are reviewing a previous answer. Original question: {question} Previous answer: {initial_answer} Task: Check the answer carefully. If it is correct, keep it. If it is wrong or incomplete, fix it. Return the corrected final answer only."""

    reflection_response = call_model_chat_completions(
        reflection_prompt,
        system=reflection_system,
        model=model,
        temperature=0.0,
    )
    reflected_answer = get_response_text(reflection_response)
    final_answer = reflected_answer if reflected_answer else initial_answer

    if too_wordy:
        print("QUESTION:", question)
        print("INITIAL ANSWER:", initial_answer)
        print("NEW REFLECTED ANSWER:", reflected_answer)
        print("FINAL ANSWER:", final_answer)

    return {
        "initial_answer": initial_answer,
        "reflected_answer": reflected_answer,
        "final_answer": final_answer,
    }

In [ ]:
#Test

example_question = "What is 17 * 26?"
reflection_result = solve_with_reflection(example_question, too_wordy=True)